In [1]:
import importlib
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)

import src.config.settings as _s
import src.data.csa_data_handler as _cdh
import src.routing.journey_planner as _jp
import tests.test_CSA as _bench

for mod in [_s, _cdh, _jp, _bench]:
    importlib.reload(mod)

from src.config.settings import get_settings
from src.routing.journey_planner import JourneyPlanner
from tests.test_CSA import (
    bench_prepare,
    bench_plan_candidates,
    bench_route,
    bench_route_backward,
)

settings = get_settings()

# Re-instantiate and re-prepare so planner uses the freshly reloaded code
planner = JourneyPlanner(settings=settings)

In [2]:
import src.config.settings

LAUSANNE_REGION_UUIDS = (
    "a7a21b73-6ffe-4fbf-a635-6e2b961f3072",
    "e168fd57-f57a-4075-a350-0dcfbb55147f",
)
REGION_UUIDS = settings.region_uuids or LAUSANNE_REGION_UUIDS

START_STOP = 8501120   # Lausanne
END_STOP   = 8501117   # Renens VD


print(f"Regions : {REGION_UUIDS}")
print(f"Stops   : {START_STOP} → {END_STOP}")

Regions : ('a7a21b73-6ffe-4fbf-a635-6e2b961f3072', 'e168fd57-f57a-4075-a350-0dcfbb55147f')
Stops   : 8501120 → 8501117


In [3]:
planner = JourneyPlanner(settings=settings)

prepare_result = bench_prepare(
    planner,
    regions=REGION_UUIDS,
    rebuild=True,
    rebuild_prerequisites=True,
)
prepare_result

"""

On Lausanne:
    v1 40 sec
    v2 40 sec -> cache everything

On whole switzerland
   ~8min
    
"""


  bench_prepare


/home/kuci/project/final/src/data/csa_data_handler.py:401: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, self.conn)



  BUILD CSA TABLES
  creating  stops...
  build_stops                     2.06s


/home/kuci/project/final/src/data/footpaths_data.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


  creating  stop_to_stop...
  build_footpaths                 2.34s
  creating  stop_times_seq...
  build_stop_times_seq            3.77s
  creating  stop_times_trips_seq...
  build_stop_times_trips_seq      5.26s
  creating  full_table_seq...
  build_full_table_seq            4.81s
  creating  connections...
  build_connections               5.66s
--------------------------------------------
  total build_all                23.90s

  FETCH DATA
  fetch_stops                     0.09s  (390 rows)
  fetch_footpaths                 0.13s  (1492 rows)
  fetch_connections              10.98s  (619910 rows)

  MATERIALIZE IN-MEMORY
  stops dict                      0.00s  (390 stops)
  footpaths dict                  0.00s  (1492 edges)
  connections loop                3.18s  (619910 rows)
  sort + columns                  3.08s  (52529 trips)
  trip meta by idx                0.03s
--------------------------------------------
  TOTAL prepare()                42.67s

----------------------

'\n\nOn Lausanne:\n    v1 40 sec\n    v2 40 sec -> cache everything\n\nOn whole switzerland\n   ~8min\n    \n'

In [4]:

# Test Sallaz → EPFL 

routes = planner.plan_candidates(
    start_stop_id=8579238,
    end_stop_id=8501214,
    travel_date="2026-05-18",
    arrival_deadline="09:00",
    max_routes=5,
)
print("Num of routes: ", len(routes))
routes

  plan_candidates     total=81.8ms  scanned=13232  routes=5
Num of routes:  5


[{'start_stop': 8579238,
  'end_stop': 8501214,
  'travel_date': '2026-05-18',
  'day': 'monday',
  'day_of_week': 'monday',
  'departure_secs': 30900,
  'arrival_secs': 32280,
  'duration_sec': 1380,
  'n_transfers': 1,
  'total_walk_m': 0,
  'steps': [{'type': 'ride',
    'trip_id': '2913.TA.91-m2-j26-1.7.R',
    'line_text': 'm2',
    'operator_id': '85:151',
    'transport': 'M',
    'from_stop': 8579238,
    'to_stop': 8591818,
    'departure_secs': 30900,
    'arrival_secs': 31380,
    'duration_sec': 480,
    'departure_time': '08:35:00',
    'arrival_time': '08:43:00'},
   {'type': 'ride',
    'trip_id': '2991.TA.91-m1-j26-1.7.R',
    'line_text': 'm1',
    'operator_id': '85:151',
    'transport': 'M',
    'from_stop': 8591818,
    'to_stop': 8501214,
    'departure_secs': 31500,
    'arrival_secs': 32280,
    'duration_sec': 780,
    'departure_time': '08:45:00',
    'arrival_time': '08:58:00'}],
  'arrival_deadline_secs': 32400,
  'departure_time': '08:35:00',
  'arrival_tim

In [5]:

# Test Sallaz → EPFL 

routes = planner.plan_candidates(
    start_stop_id=8579238,
    end_stop_id=8501214,
    travel_date="2026-05-18",
    arrival_deadline="09:00",
    max_routes=5,
)
print("Num of routes: ", len(routes))
routes

Num of routes:  5


[{'start_stop': 8579238,
  'end_stop': 8501214,
  'travel_date': '2026-05-18',
  'day': 'monday',
  'day_of_week': 'monday',
  'departure_secs': 30900,
  'arrival_secs': 32280,
  'duration_sec': 1380,
  'n_transfers': 1,
  'total_walk_m': 0,
  'steps': [{'type': 'ride',
    'trip_id': '2913.TA.91-m2-j26-1.7.R',
    'line_text': 'm2',
    'operator_id': '85:151',
    'transport': 'M',
    'from_stop': 8579238,
    'to_stop': 8591818,
    'departure_secs': 30900,
    'arrival_secs': 31380,
    'duration_sec': 480,
    'departure_time': '08:35:00',
    'arrival_time': '08:43:00'},
   {'type': 'ride',
    'trip_id': '2991.TA.91-m1-j26-1.7.R',
    'line_text': 'm1',
    'operator_id': '85:151',
    'transport': 'M',
    'from_stop': 8591818,
    'to_stop': 8501214,
    'departure_secs': 31500,
    'arrival_secs': 32280,
    'duration_sec': 780,
    'departure_time': '08:45:00',
    'arrival_time': '08:58:00'}],
  'arrival_deadline_secs': 32400,
  'departure_time': '08:35:00',
  'arrival_tim

In [8]:
from math import inf

def fmt_secs(t):
    if t == inf:
        return "INF"
    t = int(round(t))
    h = t // 3600
    m = (t % 3600) // 60
    s = t % 60
    return f"{h:02d}:{m:02d}:{s:02d}"


def forward_eat(planner, start_id, end_id, dep_time, day, max_walk_m):
    """
    Simple forward earliest-arrival CSA.

    Important:
    - Initial boarding from the start stop does NOT require min_transfer_secs.
    - Transfers after already arriving at another stop DO require min_transfer_secs.
    - Walking edges are filtered per edge by max_walk_m, same as planner.
    """
    INF = float("inf")

    arr_at = {start_id: dep_time}

    # Initial walking from origin to nearby stops
    for nbr, walk_secs, dist in planner.footpaths.get(start_id, []):
        if dist <= max_walk_m:
            t = dep_time + walk_secs
            if t < arr_at.get(nbr, INF):
                arr_at[nbr] = t

    boarded = set()

    for dep_stop, arr_stop, dep_secs, arr_secs, trip_idx, arr_adj, _ in planner.connections_by_day[day]:
        if dep_secs < dep_time:
            continue

        # CSA stopping criterion for earliest-arrival query
        if arr_at.get(end_id, INF) <= dep_secs:
            break

        if trip_idx in boarded:
            can_board = True
        elif dep_stop == start_id:
            # First boarding at origin: no transfer buffer needed
            can_board = arr_at.get(dep_stop, INF) <= dep_secs
        else:
            # Transfer boarding: require min transfer time
            can_board = arr_at.get(dep_stop, INF) + planner.min_transfer_secs <= dep_secs

        if can_board:
            boarded.add(trip_idx)

            if arr_adj < arr_at.get(arr_stop, INF):
                arr_at[arr_stop] = arr_adj

                # Walking transfer after alighting
                for nbr, walk_secs, dist in planner.footpaths.get(arr_stop, []):
                    if dist <= max_walk_m:
                        t = arr_adj + walk_secs
                        if t < arr_at.get(nbr, INF):
                            arr_at[nbr] = t

    return arr_at.get(end_id, INF)


def forward_profile(planner, start_id, end_id, t_min, t_max, day, max_walk_m, step=60, deadline=None):
    """
    Sample forward earliest-arrival CSA every `step` seconds.

    Returns Pareto frontier for:
      latest departure + earliest arrival

    If deadline is given, only keeps arrivals <= deadline.
    """
    pairs = []

    for t in range(t_min, t_max + 1, step):
        a = forward_eat(planner, start_id, end_id, t, day, max_walk_m)

        if a == float("inf"):
            continue

        if deadline is not None and a > deadline:
            continue

        pairs.append((t, a))

    return pareto_filter_latest_dep_earliest_arr(pairs)


def pareto_filter_latest_dep_earliest_arr(pairs):
    """
    Keep only non-dominated (departure, arrival) pairs.

    A pair dominates another if it departs later/equal and arrives earlier/equal.
    """
    pairs = sorted(set(pairs), key=lambda p: (-p[0], p[1]))

    frontier = []
    best_arr = float("inf")

    for dep, arr in pairs:
        if arr < best_arr:
            frontier.append((dep, arr))
            best_arr = arr

    return frontier


def route_pairs_from_plan_candidates(routes):
    return [
        (int(r["departure_secs"]), int(r["arrival_secs"]))
        for r in routes
    ]


def compare_forward_vs_backward(
    planner,
    start_id,
    end_id,
    travel_date,
    arrival_deadline,
    max_walk_m=500,
    search_window_minutes=180,
    step=60,
    max_routes=100,
):
    day = planner._day_from_travel_date(travel_date)
    deadline_secs = planner._deadline_to_relative_secs(travel_date, arrival_deadline)
    t_min = max(0, deadline_secs - search_window_minutes * 60)
    t_max = deadline_secs

    print("=" * 70)
    print("VALIDATION SETTINGS")
    print("=" * 70)
    print(f"start_id:       {start_id}")
    print(f"end_id:         {end_id}")
    print(f"travel_date:    {travel_date}")
    print(f"day:            {day}")
    print(f"deadline:       {arrival_deadline} = {fmt_secs(deadline_secs)}")
    print(f"search window:  {fmt_secs(t_min)} -> {fmt_secs(t_max)}")
    print(f"max_walk_m:     {max_walk_m}")
    print(f"step:           {step}s")
    print()

    # Forward oracle
    gold = forward_profile(
        planner=planner,
        start_id=start_id,
        end_id=end_id,
        t_min=t_min,
        t_max=t_max,
        day=day,
        max_walk_m=max_walk_m,
        step=step,
        deadline=deadline_secs,
    )

    print("=" * 70)
    print("FORWARD CSA PARETO FRONTIER / GOLD")
    print("=" * 70)
    if not gold:
        print("No valid forward routes found.")
    else:
        for d, a in gold:
            print(f"  dep {fmt_secs(d)}  arr {fmt_secs(a)}")
    print()

    # Your backward planner
    mine = planner.plan_candidates(
        start_stop_id=start_id,
        end_stop_id=end_id,
        travel_date=travel_date,
        arrival_deadline=arrival_deadline,
        max_routes=max_routes,
        max_walk_m=max_walk_m
    )

    mine_raw_pairs = route_pairs_from_plan_candidates(mine)
    mine_pareto = pareto_filter_latest_dep_earliest_arr(mine_raw_pairs)

    print("=" * 70)
    print(f"BACKWARD plan_candidates RAW ROUTES: {len(mine)}")
    print("=" * 70)
    if not mine:
        print("No backward routes found.")
    else:
        for r in mine:
            print(
                f"  dep {r['departure_time']}  arr {r['arrival_time']}  "
                f"walk={r.get('total_walk_m')}m  transfers={r.get('n_transfers')}"
            )
    print()

    print("=" * 70)
    print("BACKWARD plan_candidates AFTER PARETO FILTER")
    print("=" * 70)
    if not mine_pareto:
        print("No Pareto routes after filtering.")
    else:
        for d, a in mine_pareto:
            print(f"  dep {fmt_secs(d)}  arr {fmt_secs(a)}")
    print()

    gold_set = set(gold)
    mine_set = set(mine_pareto)

    missing_from_backward = [p for p in gold if p not in mine_set]
    extra_in_backward = [p for p in mine_pareto if p not in gold_set]

    print("=" * 70)
    print("DIFF")
    print("=" * 70)

    if not missing_from_backward and not extra_in_backward:
        print("✅ MATCH: backward Pareto routes match the forward oracle.")
    else:
        if missing_from_backward:
            print("❌ Missing from backward:")
            for d, a in missing_from_backward:
                print(f"  dep {fmt_secs(d)}  arr {fmt_secs(a)}")

        if extra_in_backward:
            print()
            print("⚠️ Extra in backward:")
            for d, a in extra_in_backward:
                print(f"  dep {fmt_secs(d)}  arr {fmt_secs(a)}")

    print()

    # Helpful debug: show suspicious extra route steps
    if extra_in_backward:
        print("=" * 70)
        print("DEBUG FIRST EXTRA BACKWARD ROUTE")
        print("=" * 70)

        extra_dep, extra_arr = extra_in_backward[0]

        for r in mine:
            if int(r["departure_secs"]) == extra_dep and int(r["arrival_secs"]) == extra_arr:
                print(f"Route dep {r['departure_time']} arr {r['arrival_time']}")
                print()

                for i, step_dict in enumerate(r["steps"], start=1):
                    print(f"Step {i}: {step_dict}")
                break

    return {
        "gold": gold,
        "mine_raw": mine_raw_pairs,
        "mine_pareto": mine_pareto,
        "missing_from_backward": missing_from_backward,
        "extra_in_backward": extra_in_backward,
        "routes": mine,
    }


# ---- RUN THE VALIDATION ----

result = compare_forward_vs_backward(
    planner=planner,
    start_id=8579238,
    end_id=8501214,
    travel_date="2026-05-18",
    arrival_deadline="09:00",
    max_walk_m=500,
    step=60,
    max_routes=100,
)

VALIDATION SETTINGS
start_id:       8579238
end_id:         8501214
travel_date:    2026-05-18
day:            monday
deadline:       09:00 = 09:00:00
search window:  06:00:00 -> 09:00:00
max_walk_m:     500
step:           60s

FORWARD CSA PARETO FRONTIER / GOLD
  dep 08:35:00  arr 08:58:00
  dep 08:30:00  arr 08:53:00
  dep 08:26:00  arr 08:50:00
  dep 08:24:00  arr 08:46:00
  dep 08:20:00  arr 08:43:00
  dep 08:17:00  arr 08:39:00
  dep 08:15:00  arr 08:38:00
  dep 08:08:00  arr 08:31:00
  dep 08:05:00  arr 08:28:00
  dep 08:02:00  arr 08:24:00
  dep 08:00:00  arr 08:23:00
  dep 07:55:00  arr 08:18:00
  dep 07:54:00  arr 08:16:00
  dep 07:51:00  arr 08:15:00
  dep 07:49:00  arr 08:13:00
  dep 07:47:00  arr 08:09:00
  dep 07:45:00  arr 08:08:00
  dep 07:38:00  arr 08:01:00
  dep 07:35:00  arr 07:58:00
  dep 07:32:00  arr 07:54:00
  dep 07:29:00  arr 07:53:00
  dep 07:28:00  arr 07:52:00
  dep 07:23:00  arr 07:46:00
  dep 07:19:00  arr 07:43:00
  dep 07:17:00  arr 07:39:00
  dep 07:15

<hr>
<hr>

In [9]:

import time

N_RUNS = 20

start_id = 8579238
end_id = 8501214
travel_date = "2026-05-18"
arrival_deadline = "09:00"
max_routes = 5
max_walk_m = 500
search_window_minutes = 180

times = []
last_mine = None

for i in range(N_RUNS):
    t0 = time.perf_counter()

    mine = planner.plan_candidates(
        start_stop_id=start_id,
        end_stop_id=end_id,
        travel_date=travel_date,
        arrival_deadline=arrival_deadline,
        max_routes=max_routes,
        max_walk_m=max_walk_m,
        search_window_minutes=search_window_minutes,
    )
    planner._route_cache.clear()

    elapsed = time.perf_counter() - t0
    times.append(elapsed)
    last_mine = mine

    print(f"Run {i + 1:02d}: {elapsed * 1000:.2f} ms | routes={len(mine)}")

avg_ms = sum(times) / len(times) * 1000
min_ms = min(times) * 1000
max_ms = max(times) * 1000

print("=" * 60)
print(f"Average over {N_RUNS} runs: {avg_ms:.2f} ms")
print(f"Min: {min_ms:.2f} ms")
print(f"Max: {max_ms:.2f} ms")
print("=" * 60)

last_mine

TypeError: JourneyPlanner.<lambda>() got an unexpected keyword argument 'search_window_minutes'

In [ ]:
import time

N_RUNS = 20

start_id = 8579238
end_id = 8501214
travel_date = "2026-05-18"
arrival_deadline = "09:00"
max_routes = 100
max_walk_m = 500
search_window_minutes = 180

times = []
last_mine = None

for i in range(N_RUNS):
    t0 = time.perf_counter()

    mine = planner.plan_candidates(
        start_stop_id=start_id,
        end_stop_id=end_id,
        travel_date=travel_date,
        arrival_deadline=arrival_deadline,
        max_routes=max_routes,
        max_walk_m=max_walk_m,
        search_window_minutes=search_window_minutes,
    )
    planner._route_cache.clear()

    elapsed = time.perf_counter() - t0
    times.append(elapsed)
    last_mine = mine

    print(f"Run {i + 1:02d}: {elapsed * 1000:.2f} ms | routes={len(mine)}")

avg_ms = sum(times) / len(times) * 1000
min_ms = min(times) * 1000
max_ms = max(times) * 1000

print("=" * 60)
print(f"Average over {N_RUNS} runs: {avg_ms:.2f} ms")
print(f"Min: {min_ms:.2f} ms")
print(f"Max: {max_ms:.2f} ms")
print("=" * 60)

last_mine

  plan_candidates     total=159.4ms  backward_calls=34  avg_backward=4.6ms  routes=33
Run 01: 159.90 ms | routes=33
Run 02: 0.01 ms | routes=33
Run 03: 0.01 ms | routes=33
Run 04: 0.00 ms | routes=33
Run 05: 0.00 ms | routes=33
Run 06: 0.00 ms | routes=33
Run 07: 0.00 ms | routes=33
Run 08: 0.00 ms | routes=33
Run 09: 0.00 ms | routes=33
Run 10: 0.00 ms | routes=33
Run 11: 0.00 ms | routes=33
Run 12: 0.00 ms | routes=33
Run 13: 0.00 ms | routes=33
Run 14: 0.00 ms | routes=33
Run 15: 0.00 ms | routes=33
Run 16: 0.00 ms | routes=33
Run 17: 0.00 ms | routes=33
Run 18: 0.00 ms | routes=33
Run 19: 0.00 ms | routes=33
Run 20: 0.00 ms | routes=33
Average over 20 runs: 8.00 ms
Min: 0.00 ms
Max: 159.90 ms


[{'start_stop': 8579238,
  'end_stop': 8501214,
  'travel_date': '2026-05-18',
  'day': 'monday',
  'day_of_week': 'monday',
  'departure_secs': 30900,
  'arrival_secs': 32280,
  'duration_sec': 1380,
  'n_transfers': 1,
  'total_walk_m': 0,
  'steps': [{'type': 'ride',
    'trip_id': '2913.TA.91-m2-j26-1.7.R',
    'line_text': 'm2',
    'operator_id': '85:151',
    'transport': 'M',
    'from_stop': 8579238,
    'to_stop': 8591818,
    'departure_secs': 30900,
    'arrival_secs': 31380,
    'duration_sec': 480,
    'departure_time': '08:35:00',
    'arrival_time': '08:43:00'},
   {'type': 'ride',
    'trip_id': '2991.TA.91-m1-j26-1.7.R',
    'line_text': 'm1',
    'operator_id': '85:151',
    'transport': 'M',
    'from_stop': 8591818,
    'to_stop': 8501214,
    'departure_secs': 31500,
    'arrival_secs': 32280,
    'duration_sec': 780,
    'departure_time': '08:45:00',
    'arrival_time': '08:58:00'}],
  'arrival_deadline_secs': 32400,
  'departure_time': '08:35:00',
  'arrival_tim